#### SubAgents

In [4]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pprint import pprint

Globals

In [5]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

Tools

In [6]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

Creating SubAgents

In [7]:
subagent_1 = create_agent(
    model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model,
    tools=[square]
)

Creating Tools that Invoke SubAgents

In [8]:
@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

Main Agent

In [9]:
main_agent = create_agent(
    model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

Agent Invocation

In [10]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='cbd681e4-eebf-438b-ba21-7958daa5f694'),
              AIMessage(content="\n\nI'll calculate the square root of 456 for you.", additional_kwargs={'usage': {'prompt_tokens': 673, 'completion_tokens': 72, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 745}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1'}, response_metadata={'usage': {'prompt_tokens': 673, 'completion_tokens': 72, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 745}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1', 'model_provider': 'bedrock', 'model_name': 'us.anthropic.claude-opus-4-6-v1'}, id='lc_run--019cde26-84d7-7851-a279-b03b5dbc9b1d-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'toolu_bdrk_01UFudAVxFZqKcnFrrJeu73Z', 'type': 'tool_call'}], invalid_tool_calls=[], usage_